# Basics

**🌐 Language:** **English** | [한국어 →](/basics-ko)

<small><em>Written by Junghyun Kim · <a href="https://github.com/jhkimon">GitHub</a> · <a href="https://www.linkedin.com/in/%EC%A0%95%ED%98%84-%EA%B9%80-883a442b2/?skipRedirect=true">LinkedIn</a></em></small>

In [1]:
from IPython.display import HTML
HTML('''<iframe width="560" height="315" src="https://www.youtube.com/embed/Ij3dd8XYSIM?si=xSMg1exSdLBjXaLK" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>''')

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

## Does Taking Medication Reduce Hospitalization Days?

> **Does medication actually help you recover faster?**

This question is closely connected to our everyday lives. It directly affects decisions such as whether the medicine I take when I am sick actually shortens my hospital stay, or which treatment a hospital should adopt as the standard of care. If we reach the wrong conclusion, we may overlook an effective treatment or continue to trust and use a treatment that does not actually work.

Now, let us consider a simple situation.

Patients with severe illness are usually sicker to begin with, so they tend to stay in the hospital longer. At the same time, they are also more likely to take the medication. In contrast, patients with mild illness are relatively less sick, so they tend to have shorter hospital stays and are less likely to take the medication.

In this situation, does the data we observe really show the “effect of the medication”? How can we use data to estimate the effect of the medication without distortion?

Let's reframe our question in the language of causal inference. Rather than simply comparing "people who took the drug vs. those who didn't," we want to compare "the same person when they took the drug vs. when they didn't." Since we can never observe both outcomes for the same person at the same time, we instead target the average effect across the entire population.

In other words, we ask: "How much does the average number of hospitalization days differ between a world where everyone takes the drug and one where no one does?"

$$
ATE = \mathbb{E}[Y_1 - Y_0]
$$

Here, $Y_1$ is the number of hospitalization days when the drug is taken, and $Y_0$ is the number when it is not. In causal inference, this is called the **Average Treatment Effect (ATE)**.

The simplest approach is to compare the average hospitalization days between those who took the drug and those who didn't. This naive comparison is valid when treatment is randomly assigned (RCT) — random assignment makes treatment independent of confounders like severity, so the two groups can be compared directly.

But our situation is different. Most people who took the drug are more severe, and most who didn't are less severe. Since the more severe people tend to be sicker to begin with, their longer hospital stays may have nothing to do with the drug — **it's severity not the drug, that's driving the difference**.

```mermaid
graph LR
    Treatment[Drug T] --> Outcome[Hospitalization Days Y]
    Severity[Severity X] --> Treatment
    Severity --> Outcome
```

A variable that affects both treatment assignment and the outcome is called a **confounder**. Confounders are what cause our estimate of the ATE to be distorted.

Let's see whether this actually causes a problem with a concrete example.

In [ ]:
drug_example = pd.DataFrame(dict(
    severity= ["M","M","M","M","M","M", "W","W","W","W"],
    drug=[1,1,1,1,1,0,  1,0,1,0],
    days=[7,7,7,7,7,8,  2,3,2,3]
))
drug_example

| Severity | Treated (T=1) | Untreated (T=0) |
| -------- | ------------: | --------------: |
| Severe   |             5 |               1 |
| Mild     |             2 |               2 |

In reality, we cannot observe both (Y_0) and (Y_1) for the same person, but for the purposes of this explanation, let’s assume we can.

* Severe patients: (Y_1 = 7, Y_0 = 8 \Rightarrow Y_1 - Y_0 = -1)
* Mild patients: (Y_1 = 2, Y_0 = 3 \Rightarrow Y_1 - Y_0 = -1)


## The Problem with Naive Comparison

Let's compute the naive comparison. The treated group average is $(5 \times 7 + 2 \times 2)/7 = 39/7$, and the untreated group average is $(1 \times 8 + 2 \times 3)/3 = 14/3$.

In [ ]:
naive_ate = drug_example.query("drug==1")["days"].mean() - drug_example.query("drug==0")["days"].mean()
print(f"Naive ATE: {naive_ate:.4f}")

$$
\hat{ATE}_{naive} = 39/7 - 14/3 \approx +0.90
$$

This result doesn't look right. The drug reduces hospitalization by 1 day for **both** men and women, so the true effect is $-1$ in every subgroup. Yet the naive comparison gives $+0.90$: the sign has flipped, making a helpful drug look **harmful**.

This distortion isn't from the drug — it comes from the **difference in group composition** (the unequal severity distribution between treated and untreated groups).

This is a textbook case of **Simpson's paradox**: a trend that holds within every subgroup can reverse once the subgroups are pooled. The drug shortens hospitalization in both the mild and severe groups, yet the combined comparison makes it look harmful — simply because most treated patients are severe patients and they stay longer regardless of the drug.

### True ATE

Since 6 severe patients each have a −1 day effect and 4 mild patients each have a −1 day effect:

$$
ATE = \frac{-1 \times 6 + (-1) \times 4}{10} = -1.0
$$

The naive comparison doesn't just shrink the effect — it **reverses its sign**, making an effective drug appear harmful.

## Solution: Inverse Probability Weighting (IPW)

The root cause of the problem is that **the severity composition of the treated and untreated groups was different**. What if we could reweight the data so that the two groups appear to have the same severity composition?

The core idea is simple. **Assign each individual a weight that makes the data look as if both groups had the same composition.** If a certain severity is overrepresented in one group, reduce their influence; if underrepresented, increase it. When severity is the source of confounding, correcting for this imbalance through reweighting makes a simple mean comparison valid.

What we want is for the severity distribution within each treatment arm to be balanced — treated and untreated groups should look the same within each severity. Since severe patients tend to take the drug more often, there are too few untreated severe patients. We need to "inflate" that underrepresented group by multiplying by the inverse of its probability.

This is the idea behind **Inverse Probability Weighting (IPW)**. Each individual receives the following weight:

$$
w =
\begin{cases}
\dfrac{1}{P(T=1 \mid X)} & \text{if } T=1 \\[6pt]
\dfrac{1}{P(T=0 \mid X)} & \text{if } T=0
\end{cases}
$$

This is the inverse of the probability of receiving the treatment the individual actually received — **rarer observations receive larger weights**.

The two cases are typically combined into a single expression:

$$
w = \frac{T}{P(T=1 \mid X)} + \frac{1-T}{P(T=0 \mid X)}
$$

When $T = 1$, only the first term survives; when $T = 0$, only the second.

### Why Does This Weight Create Balance?

The treatment probabilities by severity are:

$$
\text{Severe: } P(T=1 \mid X=M)=\frac{5}{6}, \quad P(T=0 \mid X=M)=\frac{1}{6}
$$

$$
\text{Mild: } P(T=1 \mid X=W)=\frac{1}{2}, \quad P(T=0 \mid X=W)=\frac{1}{2}
$$

The weights are the reciprocals of these values:

- Severe + treated: $6/5$, &nbsp; Severe + untreated: $6$
- Mild + treated: $2$, &nbsp; Mild + untreated: $2$

There is originally only 1 untreated for severe group, but with a weight of 6, he is **treated as if there were 6 of him**. Conversely, the 5 treated severe group each get a weight of $6/5$, so their total also sums to 6.

As a result, within the severe group, the total weight is 6 for both treated and untreated — a perfect balance. The same holds for mild patients. With this balanced dataset, a simple mean comparison recovers the true ATE:

$$
\hat{ATE}_{IPW} = \frac{7 \times 6 + 2 \times 4}{6+4} - \frac{8 \times 6 + 3 \times 4}{6+4} = -1.0
$$

In [ ]:
ps = drug_example.groupby("severity")["drug"].mean()
drug_example["ps"] = drug_example["severity"].map(ps)
drug_example["w"] = (
    drug_example["drug"] / drug_example["ps"]
    + (1 - drug_example["drug"]) / (1 - drug_example["ps"])
)

ate_ipw = (
    (drug_example["drug"] * drug_example["days"] * drug_example["w"]).sum()
    / (drug_example["drug"] * drug_example["w"]).sum()
    - ((1 - drug_example["drug"]) * drug_example["days"] * drug_example["w"]).sum()
    / ((1 - drug_example["drug"]) * drug_example["w"]).sum()
)
print(f"IPW ATE: {ate_ipw:.4f}")

The reweighted dataset is not the original observed data. It can be interpreted as a **pseudo-population** — an artificial construct in which treatment and severity appear independent. In this pseudo-population, no particular severity group is systematically over- or under-treated, recreating a situation equivalent to random assignment. As a result, a simple mean comparison is sufficient to estimate the causal effect.

## Step Further: Propensity Score

The treatment probability $e(X) = P(T=1 \mid X)$ used to construct the weights is called the **propensity score**. It represents each individual's probability of receiving the treatment given their covariates $X$.

In our example, since $X$ was a single binary variable, we could compute $P(T \mid X)$ by directly counting the treatment rate within each severity group.

In practice, however, this probability is rarely known. When there are multiple covariates or continuous variables involved, simple counting won't work. This is why we **estimate** the propensity score using a model — most commonly logistic regression.

We define the propensity score as $e(X)$ and estimate it from data as $\hat{e}(X)$. The IPW weight then becomes:

$$
\hat{w}_i =
\frac{T_i}{\hat{e}(X_i)} + \frac{1 - T_i}{1 - \hat{e}(X_i)}
$$

That said, this approach depends heavily on how well we estimate the propensity score. If the model fails to capture important nonlinearities or interactions — say, between age and severity — then $\hat{e}(X)$ may not reflect the true treatment probability. The resulting weights would be inaccurate, covariate balance may not be achieved even after reweighting, and confounding may not be fully removed.

## References

This notebook draws on Hernán & Robins (2020), Matheus Facure's *Causal Inference for the Brave and True*, and the Korean translation by CausalInferenceLab.

- **Hernán, M. A., & Robins, J. M. (2020)**. *Causal Inference: What If*. Chapman & Hall/CRC.  
  [https://miguelhernan.org/whatifbook](https://miguelhernan.org/whatifbook)

- **Facure, M. (2022)**. *Causal Inference for the Brave and True*. Online book.  
  [https://matheusfacure.github.io/python-causality-handbook/](https://matheusfacure.github.io/python-causality-handbook/)

- **CausalInferenceLab**. *Causal Inference for the Brave and True* Korean translation.  
  [https://causalinferencelab.github.io/Causal-Inference-with-Python/](https://causalinferencelab.github.io/Causal-Inference-with-Python/)